# PicoGPT MathNet 6.62M training\n\nChoose **Runtime → Change runtime type → GPU** before running. This notebook writes every artifact to Google Drive and trains a new isolated package; it does not modify PicoGPT's original files.\n\nBefore opening in Colab, commit/push this `experiments/mathnet_6_6m/` directory to the repository (or change `REPO_URL` below to a fork/branch containing it).

In [ ]:
from google.colab import drive\nfrom pathlib import Path\nimport subprocess\n\ndrive.mount('/content/drive')\nREPO_URL = 'https://github.com/PranavBansal7/PicoGPT.git'\nBRANCH = 'main'  # Change if the isolated experiment is pushed on a feature branch.\nPROJECT_DIR = Path('/content/PicoGPT')\nVARIANT = 'pre_rms_mha'  # post_ln_mha, pre_ln_mha, pre_rms_mha, or pre_rms_gqa\nSEED = 1337\nRUN_NAME = f'{VARIANT}_seed{SEED}'\nEXPERIMENT_ROOT = Path('/content/drive/MyDrive/PicoGPT_MathNet')\nRUN_DIR = EXPERIMENT_ROOT / 'runs' / RUN_NAME\nDATA_DIR = EXPERIMENT_ROOT / 'data'  # Shared immutable prepared corpus for every variant.\n\nif not PROJECT_DIR.exists():\n    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)\nsubprocess.run(['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], check=True)\nassert (PROJECT_DIR / 'experiments/mathnet_6_6m/model.py').exists(), (\n    'The clone does not contain this experiment. Push the new experiment directory or set REPO_URL/BRANCH.')\nRUN_DIR.mkdir(parents=True, exist_ok=True)\nprint(f'Project: {PROJECT_DIR}\\nRun artifacts: {RUN_DIR}')

In [ ]:
%cd /content/PicoGPT\n!pip -q install -r experiments/mathnet_6_6m/requirements-colab.txt\n!python -m unittest experiments.mathnet_6_6m.test_model\n!nvidia-smi

In [ ]:
# Downloads the pinned public MathNet revision, creates an internal 80/10/10 split,\n# trains BPE on internal-train only, and records a manifest/hashes under Drive.\nfrom pathlib import Path\nimport subprocess\nif not (DATA_DIR / 'data_metadata.json').exists():\n    DATA_DIR.mkdir(parents=True, exist_ok=True)\n    subprocess.run([\n        'python', '-m', 'experiments.mathnet_6_6m.prepare_data',\n        '--output-dir', str(DATA_DIR),\n    ], cwd=PROJECT_DIR, check=True)\nelse:\n    print(f'Reusing prepared data: {DATA_DIR}')

In [ ]:
# 41 logical epochs × 100 optimizer updates = exactly 4,100 updates.\n# To resume after a Colab interruption, add:\n# '--resume', str(RUN_DIR / 'checkpoints/last.pt')\nsubprocess.run([\n    'python', '-m', 'experiments.mathnet_6_6m.train',\n    '--data-dir', str(DATA_DIR), '--run-dir', str(RUN_DIR),\n    '--epochs', '41', '--steps-per-epoch', '100', '--variant', VARIANT, '--seed', str(SEED),\n    '--micro-batch-size', '16', '--gradient-accumulation-steps', '4',\n], cwd=PROJECT_DIR, check=True)

In [ ]:
# Test only once after selection by validation loss; then save samples and a factual report.\nsubprocess.run([\n    'python', '-m', 'experiments.mathnet_6_6m.evaluate',\n    '--data-dir', str(DATA_DIR), '--run-dir', str(RUN_DIR), '--test-batches', '500',\n], cwd=PROJECT_DIR, check=True)\nsubprocess.run(['python', '-m', 'experiments.mathnet_6_6m.generate', '--run-dir', str(RUN_DIR)], cwd=PROJECT_DIR, check=True)\nsubprocess.run(['python', '-m', 'experiments.mathnet_6_6m.benchmark_decode', '--run-dir', str(RUN_DIR)], cwd=PROJECT_DIR, check=True)\nsubprocess.run(['python', '-m', 'experiments.mathnet_6_6m.report', '--run-dir', str(RUN_DIR)], cwd=PROJECT_DIR, check=True)\nprint((RUN_DIR / 'RESULTS.md').read_text())